# Exercise 2 — Data Quality Probe

**Decision scenario:** Validate a back-test before investment-committee review by identifying empty histories, suspicious date ranges, corporate actions, symbol discontinuities, delistings, and structural changes.

**Sources**
1. Yahoo Finance via `yfinance`: provider-reported names, currencies, daily OHLCV, dividends, and split fields.
2. WRDS/CRSP: independent professional-source validation of security identities, historical coverage, prices/returns, and distribution records.

Yahoo Finance is the primary automated probe. WRDS/CRSP is used to distinguish a true absence of trading data from a provider or ticker-symbol discontinuity.

In [46]:
import pandas as pd
import numpy as np
import yfinance as yf
import wrds

from datetime import date

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

In [47]:
START = "2022-06-01"
END = date.today().isoformat()
RETRIEVAL_DATE = date.today().isoformat()

TICKERS = ["NVDA", "GE", "MMM", "JNJ", "ATVI", "VMW", "GOOG", "META", "KVUE", "SOLV"]

event_evidence = pd.DataFrame([
    ["NVDA", "Stock split", "10-for-1 forward split", "2024-06-10",
     "NVIDIA investor-relations earnings release / stock-split announcement"],
    ["GE", "Corporate separation", "GE Vernova separation; GE continued as GE Aerospace", "2024-04-02",
     "GE investor-relations announcement on GE Vernova separation"],
    ["MMM", "Spinoff", "3M completed the Solventum health-care spinoff", "2024-04-01",
     "3M investor-relations announcement on Solventum spinoff"],
    ["JNJ", "Separation / exchange offer", "Johnson & Johnson completed the Kenvue exchange offer", "2023-08-23",
     "Johnson & Johnson investor-relations announcement"],
    ["ATVI", "Acquisition and delisting", "Microsoft completed its acquisition of Activision Blizzard", "2023-10-13",
     "Microsoft acquisition-closing announcement"],
    ["VMW", "Acquisition and delisting", "Broadcom completed its acquisition of VMware", "2023-11-22",
     "Broadcom acquisition-closing announcement"],
    ["GOOG", "Stock split", "Alphabet completed a 20-for-1 forward split", "2022-07-18",
     "Alphabet investor-relations release / SEC filing"],
    ["META", "Company and ticker change", "Facebook became Meta Platforms; FB changed to META", "2022-06-09",
     "Meta investor-relations or Nasdaq notice"],
    ["KVUE", "New listing / separation", "Kenvue began trading in 2023 and was separated from J&J", "2023-05-04",
     "Kenvue investor-relations IPO announcement; J&J exchange-offer announcement"],
    ["SOLV", "New listing / spinoff", "Solventum began trading after its 3M spinoff", "2024-04-01",
     "3M/Solventum investor-relations separation announcement"],
], columns=["ticker", "event_type", "what_happened", "effective_date", "authoritative_source"])

display(event_evidence)

,ticker,event_type,what_happened,effective_date,authoritative_source
0,NVDA,Stock split,10-for-1 forward split,2024-06-10,NVIDIA investor-relations earnings release / s...
1,GE,Corporate separation,GE Vernova separation; GE continued as GE Aero...,2024-04-02,GE investor-relations announcement on GE Verno...
2,MMM,Spinoff,3M completed the Solventum health-care spinoff,2024-04-01,3M investor-relations announcement on Solventu...
3,JNJ,Separation / exchange offer,Johnson & Johnson completed the Kenvue exchang...,2023-08-23,Johnson & Johnson investor-relations announcement
4,ATVI,Acquisition and delisting,Microsoft completed its acquisition of Activis...,2023-10-13,Microsoft acquisition-closing announcement
5,VMW,Acquisition and delisting,Broadcom completed its acquisition of VMware,2023-11-22,Broadcom acquisition-closing announcement
6,GOOG,Stock split,Alphabet completed a 20-for-1 forward split,2022-07-18,Alphabet investor-relations release / SEC filing
7,META,Company and ticker change,Facebook became Meta Platforms; FB changed to ...,2022-06-09,Meta investor-relations or Nasdaq notice
8,KVUE,New listing / separation,Kenvue began trading in 2023 and was separated...,2023-05-04,Kenvue investor-relations IPO announcement; J&...
9,SOLV,New listing / spinoff,Solventum began trading after its 3M spinoff,2024-04-01,3M/Solventum investor-relations separation ann...


## Four validation classes

1. **Coverage validation:** empty histories, first/last dates, and suspicious gaps.
2. **Corporate-action validation:** splits and dividends that can create false returns.
3. **Security-identity validation:** provider company name, ticker/name changes, and historical identifier continuity.
4. **Structural-event validation:** acquisitions, delistings, spinoffs, and newly listed securities.

An empty or failed provider response is a flag requiring investigation. It is not proof of a delisting or an economic loss.

In [48]:
def yahoo_probe(ticker, start=START, end=END):
    """Fault-tolerant Yahoo Finance data-quality probe for one ticker."""
    
    result = {
        "ticker": ticker,
        "requested_start": start,
        "requested_end": end,
        "rows_returned": 0,
        "history_empty": True,
        "first_available_date": None,
        "last_available_date": None,
        "latest_close": np.nan,
        "currency": None,
        "provider_company_name": None,
        "detected_splits": "None detected",
        "detected_dividends": "None detected",
        "large_gap_count": 0,
        "missing_date_or_likely_delisting_flag": None,
        "yahoo_error": None,
    }

    try:
        security = yf.Ticker(ticker)

        # yfinance treats end as exclusive, so add one day.
        end_for_download = (
            pd.Timestamp(end) + pd.Timedelta(days=1)
        ).strftime("%Y-%m-%d")

        history = security.history(
            start=start,
            end=end_for_download,
            auto_adjust=False,
            actions=True
        )

        # Metadata is useful but should not stop the notebook if unavailable.
        try:
            info = security.get_info()
            result["provider_company_name"] = (
                info.get("longName") or info.get("shortName")
            )
            result["currency"] = info.get("currency")
        except Exception as metadata_error:
            result["yahoo_error"] = f"Metadata issue: {metadata_error}"

        if history.empty:
            result["missing_date_or_likely_delisting_flag"] = (
                "EMPTY HISTORY — investigate provider coverage, ticker identity, "
                "or delisting independently; do not infer an event from emptiness alone."
            )
            return result

        history.index = pd.to_datetime(history.index).tz_localize(None)
        history = history.sort_index()

        result["rows_returned"] = len(history)
        result["history_empty"] = False
        result["first_available_date"] = history.index.min().date()
        result["last_available_date"] = history.index.max().date()
        valid_closes = history["Close"].dropna()
        if not valid_closes.empty:
            result["latest_close"] = round(float(valid_closes.iloc[-1]), 2)

        splits = history.loc[
            history["Stock Splits"].fillna(0) != 0, "Stock Splits"
        ]
        dividends = history.loc[
            history["Dividends"].fillna(0) != 0, "Dividends"
        ]

        if not splits.empty:
            result["detected_splits"] = "; ".join(
                f"{idx.date()} (factor {value:g})"
                for idx, value in splits.items()
            )

        if not dividends.empty:
            result["detected_dividends"] = "; ".join(
                f"{idx.date()} (${value:.4f})"
                for idx, value in dividends.items()
            )

        expected_start = pd.Timestamp(start)
        expected_end = min(pd.Timestamp(end), pd.Timestamp.today().normalize())

        flags = []

        # Allows normal weekends and holiday closures.
        if history.index.min() > expected_start + pd.Timedelta(days=7):
            flags.append("late first date")

        if history.index.max() < expected_end - pd.Timedelta(days=7):
            flags.append("early last date / possible delisting or symbol discontinuity")

        gaps = history.index.to_series().diff().dt.days
        result["large_gap_count"] = int((gaps > 7).sum())

        if result["large_gap_count"] > 0:
            flags.append(f"{result['large_gap_count']} gap(s) over 7 calendar days")

        result["missing_date_or_likely_delisting_flag"] = (
            "; ".join(flags) if flags else "No major coverage flag"
        )

    except Exception as error:
        result["yahoo_error"] = str(error)
        result["missing_date_or_likely_delisting_flag"] = (
            "FAILED RESPONSE — investigate provider/API issue; do not classify "
            "as a delisting without corroborating evidence."
        )

    return result

In [49]:
yahoo_summary = pd.DataFrame(
    [yahoo_probe(ticker) for ticker in TICKERS]
)

display(yahoo_summary)

yahoo_summary.to_csv("exercise2_yahoo_probe.csv", index=False)

/var/folders/_s/682h57nj4f9b8mhphbnjfmg80000gn/T/ipykernel_83649/417981576.py:27: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pd.Timestamp(end) + pd.Timedelta(days=1)
/var/folders/_s/682h57nj4f9b8mhphbnjfmg80000gn/T/ipykernel_83649/417981576.py:90: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  if history.index.min() > expected_start + pd.Timedelta(days=7):
/var/folders/_s/682h57nj4f9b8mhphbnjfmg80000gn/T/ipykernel_83649/417981576.py:93: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  i

,ticker,requested_start,requested_end,rows_returned,history_empty,first_available_date,last_available_date,latest_close,currency,provider_company_name,detected_splits,detected_dividends,large_gap_count,missing_date_or_likely_delisting_flag,yahoo_error
0,NVDA,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,223.67,USD,NVIDIA Corporation,2024-06-10 (factor 10),2022-06-08 ($0.0040); 2022-09-07 ($0.0040); 20...,0,No major coverage flag,None
1,GE,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,325.42,USD,GE Aerospace,2023-01-04 (factor 1.281); 2024-04-02 (factor ...,2022-06-27 ($0.0498); 2022-09-26 ($0.0498); 20...,0,No major coverage flag,None
2,MMM,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,164.40,USD,3M Company,2024-04-01 (factor 1.196),2022-08-19 ($1.2458); 2022-11-17 ($1.2458); 20...,0,No major coverage flag,None
3,JNJ,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,267.08,USD,Johnson & Johnson,None detected,2022-08-22 ($1.1300); 2022-11-21 ($1.1300); 20...,0,No major coverage flag,None
4,ATVI,2022-06-01,2026-09-10,0,True,None,None,NaN,None,None,None detected,None detected,0,"EMPTY HISTORY — investigate provider coverage,...",None
5,VMW,2022-06-01,2026-09-10,0,True,None,None,NaN,None,None,None detected,None detected,0,"EMPTY HISTORY — investigate provider coverage,...",None
6,GOOG,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,328.38,USD,Alphabet Inc.,2022-07-18 (factor 20),2024-06-10 ($0.2000); 2024-09-09 ($0.2000); 20...,0,No major coverage flag,None
7,META,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,653.69,USD,"Meta Platforms, Inc.",None detected,2024-02-21 ($0.5000); 2024-06-14 ($0.5000); 20...,0,No major coverage flag,None
8,KVUE,2022-06-01,2026-09-10,841,False,2023-05-04,2026-09-10,17.75,USD,Kenvue Inc.,None detected,2023-08-25 ($0.2000); 2023-11-07 ($0.2000); 20...,0,late first date,None
9,SOLV,2022-06-01,2026-09-10,617,False,2024-03-26,2026-09-10,90.03,USD,Solventum Corporation,None detected,None detected,0,late first date,None


In [50]:
def wrds_probe(ticker, start=START, end=END):
    """Independent WRDS/CRSP check. Returns an error message rather than crashing."""
    
    result = {
        "ticker": ticker,
        "wrds_name_records": 0,
        "wrds_permnos": None,
        "wrds_company_names": None,
        "wrds_daily_rows": 0,
        "wrds_first_date": None,
        "wrds_last_date": None,
        "wrds_latest_close": np.nan,
        "wrds_delisting_records": 0,
        "wrds_error": None,
    }

    try:
        ticker_sql = ticker.replace("'", "''")

        names_sql = f"""
            SELECT permno, ticker, comnam, namedt, nameenddt
            FROM crsp.stocknames
            WHERE ticker = '{ticker_sql}'
              AND namedt <= '{end}'
              AND nameenddt >= '{start}'
            ORDER BY permno, namedt
        """

        names = db.raw_sql(names_sql, date_cols=["namedt", "nameenddt"])

        if names.empty:
            result["wrds_error"] = "No CRSP stocknames record returned for this ticker/window."
            return result

        permnos = names["permno"].dropna().astype(int).unique().tolist()

        result["wrds_name_records"] = len(names)
        result["wrds_permnos"] = ", ".join(map(str, permnos))
        result["wrds_company_names"] = "; ".join(
            names["comnam"].dropna().astype(str).unique()
        )

        permno_list = ",".join(map(str, permnos))

        daily_sql = f"""
            SELECT permno, date, prc, ret
            FROM crsp.dsf
            WHERE permno IN ({permno_list})
              AND date BETWEEN '{start}' AND '{end}'
            ORDER BY permno, date
        """

        daily = db.raw_sql(daily_sql, date_cols=["date"])

        if not daily.empty:
            daily["close_abs"] = daily["prc"].abs()
            result["wrds_daily_rows"] = len(daily)
            result["wrds_first_date"] = daily["date"].min().date()
            result["wrds_last_date"] = daily["date"].max().date()
            result["wrds_latest_close"] = round(
                float(daily.sort_values("date")["close_abs"].iloc[-1]), 2
            )

        delisting_sql = f"""
            SELECT permno, dlstdt, dlstcd, dlret
            FROM crsp.dsedelist
            WHERE permno IN ({permno_list})
              AND dlstdt BETWEEN '{start}' AND '{end}'
            ORDER BY permno, dlstdt
        """

        delisting = db.raw_sql(delisting_sql, date_cols=["dlstdt"])
        result["wrds_delisting_records"] = len(delisting)

    except Exception as error:
        result["wrds_error"] = str(error)

    return result

In [51]:
try:
    db = wrds.Connection()

    wrds_summary = pd.DataFrame(
        [wrds_probe(ticker) for ticker in TICKERS]
    )

except Exception as connection_error:
    wrds_summary = pd.DataFrame({
        "ticker": TICKERS,
        "wrds_error": [f"WRDS connection failed: {connection_error}"] * len(TICKERS)
    })

display(wrds_summary)

wrds_summary.to_csv("exercise2_wrds_validation.csv", index=False)

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


,ticker,wrds_name_records,wrds_permnos,wrds_company_names,wrds_daily_rows,wrds_first_date,wrds_last_date,wrds_latest_close,wrds_delisting_records,wrds_error
0,NVDA,1,86580,NVIDIA CORP,650,2022-06-01,2024-12-31,134.29,1,None
1,GE,1,12060,GENERAL ELECTRIC CO,650,2022-06-01,2024-12-31,166.79,1,None
2,MMM,3,22592,3M CO,650,2022-06-01,2024-12-31,129.09,1,None
3,JNJ,1,22111,JOHNSON & JOHNSON,650,2022-06-01,2024-12-31,144.62,1,None
4,ATVI,1,79678,ACTIVISION BLIZZARD INC,344,2022-06-01,2023-10-12,94.42,1,None
5,VMW,2,92257,VMWARE INC,372,2022-06-01,2023-11-21,142.48,1,None
6,GOOG,1,14542,ALPHABET INC,650,2022-06-01,2024-12-31,190.44,1,None
7,META,1,13407,META PLATFORMS INC,650,2022-06-01,2024-12-31,585.51,1,None
8,KVUE,1,23876,KENVUE INC,418,2023-05-04,2024-12-31,21.35,1,None
9,SOLV,1,24877,SOLVENTUM CORP,191,2024-04-01,2024-12-31,66.06,1,None


In [52]:
delisting_details = []

for _, row in wrds_summary.iterrows():
    if pd.isna(row["wrds_permnos"]) or row["wrds_permnos"] is None:
        continue

    permno_list = row["wrds_permnos"]

    query = f"""
        SELECT permno, dlstdt, dlstcd, dlret
        FROM crsp.dsedelist
        WHERE permno IN ({permno_list})
        ORDER BY permno, dlstdt
    """

    detail = db.raw_sql(query, date_cols=["dlstdt"])
    detail["ticker"] = row["ticker"]
    delisting_details.append(detail)

delisting_details = pd.concat(delisting_details, ignore_index=True)

display(delisting_details)

delisting_details.to_csv("exercise2_crsp_delisting_details.csv", index=False)

/var/folders/_s/682h57nj4f9b8mhphbnjfmg80000gn/T/ipykernel_83649/16388194.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  delisting_details = pd.concat(delisting_details, ignore_index=True)


,permno,dlstdt,dlstcd,dlret,ticker
0,86580,2024-12-31,100,<NA>,NVDA
1,12060,2024-12-31,100,<NA>,GE
2,22592,2024-12-31,100,<NA>,MMM
3,22111,2024-12-31,100,<NA>,JNJ
4,79678,2023-10-12,233,0.006143,ATVI
5,92257,2023-11-21,241,0.374743,VMW
6,14542,2024-12-31,100,<NA>,GOOG
7,13407,2024-12-31,100,<NA>,META
8,23876,2024-12-31,100,<NA>,KVUE
9,24877,2024-12-31,100,<NA>,SOLV


In [53]:
final_probe = (
    yahoo_summary
    .merge(wrds_summary, on="ticker", how="left")
    .merge(event_evidence, on="ticker", how="left")
)

display(final_probe)

final_probe.to_csv("exercise2_final_probe.csv", index=False)

,ticker,requested_start,requested_end,rows_returned,history_empty,first_available_date,last_available_date,latest_close,currency,provider_company_name,detected_splits,detected_dividends,large_gap_count,missing_date_or_likely_delisting_flag,yahoo_error,wrds_name_records,wrds_permnos,wrds_company_names,wrds_daily_rows,wrds_first_date,wrds_last_date,wrds_latest_close,wrds_delisting_records,wrds_error,event_type,what_happened,effective_date,authoritative_source
0,NVDA,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,223.67,USD,NVIDIA Corporation,2024-06-10 (factor 10),2022-06-08 ($0.0040); 2022-09-07 ($0.0040); 20...,0,No major coverage flag,None,1,86580,NVIDIA CORP,650,2022-06-01,2024-12-31,134.29,1,None,Stock split,10-for-1 forward split,2024-06-10,NVIDIA investor-relations earnings release / s...
1,GE,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,325.42,USD,GE Aerospace,2023-01-04 (factor 1.281); 2024-04-02 (factor ...,2022-06-27 ($0.0498); 2022-09-26 ($0.0498); 20...,0,No major coverage flag,None,1,12060,GENERAL ELECTRIC CO,650,2022-06-01,2024-12-31,166.79,1,None,Corporate separation,GE Vernova separation; GE continued as GE Aero...,2024-04-02,GE investor-relations announcement on GE Verno...
2,MMM,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,164.40,USD,3M Company,2024-04-01 (factor 1.196),2022-08-19 ($1.2458); 2022-11-17 ($1.2458); 20...,0,No major coverage flag,None,3,22592,3M CO,650,2022-06-01,2024-12-31,129.09,1,None,Spinoff,3M completed the Solventum health-care spinoff,2024-04-01,3M investor-relations announcement on Solventu...
3,JNJ,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,267.08,USD,Johnson & Johnson,None detected,2022-08-22 ($1.1300); 2022-11-21 ($1.1300); 20...,0,No major coverage flag,None,1,22111,JOHNSON & JOHNSON,650,2022-06-01,2024-12-31,144.62,1,None,Separation / exchange offer,Johnson & Johnson completed the Kenvue exchang...,2023-08-23,Johnson & Johnson investor-relations announcement
4,ATVI,2022-06-01,2026-09-10,0,True,None,None,NaN,None,None,None detected,None detected,0,"EMPTY HISTORY — investigate provider coverage,...",None,1,79678,ACTIVISION BLIZZARD INC,344,2022-06-01,2023-10-12,94.42,1,None,Acquisition and delisting,Microsoft completed its acquisition of Activis...,2023-10-13,Microsoft acquisition-closing announcement
5,VMW,2022-06-01,2026-09-10,0,True,None,None,NaN,None,None,None detected,None detected,0,"EMPTY HISTORY — investigate provider coverage,...",None,2,92257,VMWARE INC,372,2022-06-01,2023-11-21,142.48,1,None,Acquisition and delisting,Broadcom completed its acquisition of VMware,2023-11-22,Broadcom acquisition-closing announcement
6,GOOG,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,328.38,USD,Alphabet Inc.,2022-07-18 (factor 20),2024-06-10 ($0.2000); 2024-09-09 ($0.2000); 20...,0,No major coverage flag,None,1,14542,ALPHABET INC,650,2022-06-01,2024-12-31,190.44,1,None,Stock split,Alphabet completed a 20-for-1 forward split,2022-07-18,Alphabet investor-relations release / SEC filing
7,META,2022-06-01,2026-09-10,1073,False,2022-06-01,2026-09-10,653.69,USD,"Meta Platforms, Inc.",None detected,2024-02-21 ($0.5000); 2024-06-14 ($0.5000); 20...,0,No major coverage flag,None,1,13407,META PLATFORMS INC,650,2022-06-01,2024-12-31,585.51,1,None,Company and ticker change,Facebook became Meta Platforms; FB changed to ...,2022-06-09,Meta investor-relations or Nasdaq notice
8,KVUE,2022-06-01,2026-09-10,841,False,2023-05-04,2026-09-10,17.75,USD,Kenvue Inc.,None detected,2023-08-25 ($0.2000); 2023-11-07 ($0.2000); 20...,0,late first date,None,1,23876,KENVUE INC,418,2023-05-04,2024-12-31,21.35,1,None,New listing / separation,Kenvue began trading in 2023 and was separated...,2023-05-04,Kenvue investor-relations IPO announcement; J&...
9,SOLV,2022-06-01,2026-09-10,617,False,2024-03-26,2026-09-10,90.03,USD,Solventum Corporation,None detected,None detected,0,late first date,None,1,24877,SOLVENTUM CORP,191,2024-04-01,2024-12-31,66.06,1,None,New listing / spinoff,Solv

## Findings

The Yahoo Finance probe returned full June 1, 2022-to-retrieval-date coverage for NVDA, GE, MMM, JNJ, GOOG, and META. It returned empty histories for ATVI and VMW, despite CRSP retaining their historical observations through October 12, 2023 and November 21, 2023, respectively. These are provider/symbol discontinuities after completed acquisitions, not evidence that shareholders lost their entire investment.

KVUE begins on May 4, 2023 because it was not publicly traded before that date. SOLV begins on March 26, 2024 in Yahoo Finance but April 1, 2024 in CRSP, which requires the analyst to distinguish possible when-issued/provider coverage from the regular-way spinoff date.

Yahoo Finance identified ordinary forward split factors for NVDA and GOOG. Its split/action field also contains factors for GE and MMM around their corporate separations. These must not be treated as ordinary stock splits without checking the corporate-event evidence; they reflect structural changes that can create false price-only returns.

CRSP coverage ends on December 31, 2024 for several securities that remain active. I treat this as a CRSP extract/file-end coverage limitation, not a delisting. Yahoo Finance is therefore used for the current-price field, while CRSP is used to validate historical identity, coverage, and acquisition-related termination.

## Source log

| Source | Access route | Purpose | Retrieval date | Limitation |
|---|---|---|---|---|
| Yahoo Finance | Python `yfinance`; `history(auto_adjust=False, actions=True)` and `get_info()` | Daily history, provider name, currency, close, dividends, and splits | `RETRIEVAL_DATE` | A ticker can resolve to incomplete, renamed, or successor-listing data. |
| WRDS / CRSP | Python `wrds.Connection()` and CRSP tables | Historical names, PERMNOs, daily observations, and delisting records | `RETRIEVAL_DATE` | Requires successful Purdue WRDS access; ticker is not a permanent identifier. |
| Official company/acquirer/exchange materials | Event-evidence register | Corroborate event type and effective date | `RETRIEVAL_DATE` | Explains the event but does not replace return-data validation. |